# ICU Mortality Prediction Model — **Phase 2**
### Feature engineering: comorbidities, physiological ratios, secondary diagnoses, and Table 1

This notebook continues directly from where Phase 1 stopped. The base `modelo_A_preparado.csv`
(**5,676** unique-pair index admissions **or recovered by positional alignment**, with known hospital
outcome, SAPS3 already reserved as the yardstick, and the nearly empty first-hour blood gas already
dropped) is the starting point.

**Special attention in this phase:** Phase 1 recovered 551 of the 555 originally ambiguous index
admissions by positional alignment (not by key) — but that recovery was only tested against the four
predictor sheets already integrated (Folha3–Folha6). Folha8, entering now for the first time, needs the
**same position verification**, independently — and, as we will see in Section 4, it reveals a small
problem of its own, specific to this sheet, affecting a handful of patients who were not even part of
the original 555.

Four tasks, in script order:

1. **Comorbidities and physiological ratios.** The ~53 indicators of Folha3 become a total count and
   eleven clinical groups; two physiological ratios (shock index, urea/creatinine) are derived from the
   6-hour extremes.
2. **Secondary diagnoses.** Folha8 (not yet integrated) enters now — by **positional alignment**, not by
   key, with the same safety verification used in Phase 1 — only diagnoses flagged as *present at ICU
   admission* (the cut the temporal contract allows), reduced to a count and eleven major system groups.
3. **Missingness indicators.** Created for laboratory and physiology variables, guided by the clinical
   informativeness of absence — not by a percentage cutoff — with a safety check against
   outcome-dependent missingness.
4. **Table 1 and the two variable sets.** The descriptive table by outcome, and the column lists for
   Set 1 (ready-made scores + demographics/context) and Set 2 (raw components + demographics/context).

## 1 · Load the Phase 1 base and Folha8 (not yet integrated)

Two files enter here: `modelo_A_preparado.csv` (output of Phase 1, Section 10) and the original Excel,
only to read Folha8 — the one sheet Phase 1 deliberately left out.

In [ ]:
import os, pandas as pd, numpy as np
from scipy import stats
pd.set_option('display.max_columns', 60)

# === Configuration ===
MODELO_A_PATH = 'outputs/phase1/modelo_A_preparado.csv'   # output of Phase 1 (Section 10)
FILE_PATH     = 'data/Excel01GlobalAdmission.xlsx'
MERGE_KEYS    = ['EpimedCode', 'MedicalRecord']
OUTDIR        = 'outputs/phase2'
os.makedirs(OUTDIR, exist_ok=True)

modelo_a = pd.read_csv(MODELO_A_PATH, encoding='utf-8-sig')
modelo_a['adm_datetime'] = pd.to_datetime(modelo_a['adm_datetime'], errors='coerce')
print('Base da Fase 1 carregada:', modelo_a.shape)
print(f"Hospital deaths: {int(modelo_a['mortalidade_hospitalar'].sum())} "
      f"({modelo_a['mortalidade_hospitalar'].mean()*100:.1f}%)")

In [ ]:
# Folha8 (secondary diagnoses) — the only sheet not yet integrated in Phase 1.
# The real header is on the 2nd row, as in the other sheets.
folha8 = pd.read_excel(FILE_PATH, sheet_name='Folha8', header=1)
folha8['_linha_bruta'] = folha8.index

# Same boolean cleaning as Phase 1, applied here only to Folha8.
BOOL_MAP = {'verdadeiro':1,'falso':0,'sim':1,'no':0,'nao':0,'true':1,'false':0}
def to_bool(series):
    s = series.astype('string').str.strip().str.lower()
    return s.map(BOOL_MAP).astype('Float64')
def clean_booleans(df, protect):
    out = df.copy()
    for col in out.columns:
        if col in protect: continue
        vals = out[col].dropna().astype('string').str.strip().str.lower().unique()
        if len(vals) and set(vals).issubset(BOOL_MAP.keys()):
            out[col] = to_bool(out[col])
    return out
folha8 = clean_booleans(folha8, protect=set(MERGE_KEYS) | {'_linha_bruta'})
print(f'Folha8 carregada: {len(folha8)} linhas.')

# Propagate Phase 1's 14 exact-duplicate removals to Folha8.
# The Phase 1 CSV no longer contains those positions — but Folha8 still does.
# Without this removal, the position-based merge could associate the wrong row.
_linhas_coorte = set(
    pd.read_csv(MODELO_A_PATH, encoding='utf-8-sig',
                usecols=['_linha_bruta'])['_linha_bruta'])
n_folha8_antes = len(folha8)
folha8 = folha8[folha8['_linha_bruta'].isin(_linhas_coorte)].reset_index(drop=True)
print(f'Folha8 after removing positions absent from the cohort: '
      f'{len(folha8)} linhas ({n_folha8_antes - len(folha8)} removidas).')

# Alignment verification BEFORE the merge:
# for each cohort row, confirm that the corresponding position
# in Folha8 carries the same key. Detects isolated transpositions.
folha8_chave_por_linha = folha8.set_index('_linha_bruta')[MERGE_KEYS]
alvo = folha8_chave_por_linha.reindex(modelo_a['_linha_bruta']).reset_index(drop=True)
bate = ((alvo['EpimedCode'].values    == modelo_a['EpimedCode'].values) &
        (alvo['MedicalRecord'].values == modelo_a['MedicalRecord'].values))
modelo_a['folha8_integravel'] = bate

n_ok    = int(bate.sum())
n_falha = int((~bate).sum())
print(f'\nPositional alignment verification (Folha8):')
print(f'  Alinhadas corretamente: {n_ok} de {len(modelo_a)}')
if n_falha:
    print(f'  ⚠️  {n_falha} case(s) misaligned — diagnoses will be zeroed:')
    print(modelo_a.loc[~bate, MERGE_KEYS + ['_linha_bruta']].to_string(index=False))
    desalinhados_path = f'{OUTDIR}/fase2_folha8_desalinhamentos.csv'
    modelo_a.loc[~bate, MERGE_KEYS + ['_linha_bruta']].to_csv(
        desalinhados_path, index=False, encoding='utf-8-sig')
    print(f'  Lista salva: {desalinhados_path}')
else:
    print('  OK — nenhum desalinhamento detectado.')

## 2 · Comorbidities: total count and clinical groups (Folha3)

The ~53 boolean comorbidity indicators already integrated by Phase 1 (Folha3) become two things: a
**total count** — the simple sum of flags, as "comorbidity burden" — and **eleven clinical groups**,
grouped by system (cardiovascular, renal, hepatic, oncological, chronic respiratory, metabolic/
endocrine, immunosuppression, transplant, neurological, psychiatric/substance use, and a residual group).

Residual blanks (~0.1% per flag) are treated as "not recorded" = absent/0 — the same convention already
used in Phase 1 for the other booleans. This differs from the missingness-indicator logic of Section 4:
here absence is data-entry noise in a checklist field, not a clinical signal of a test not drawn.

In [ ]:
COMORB_GRUPOS = {
    'comorb_cardiovascular': ['IsChfNyhaClass23','IsChfNyhaClass4','IsArterialHypertension',
        'IsAngina','IsPreviousAMI','IsCardiacArrhythmia','IsChronicAtrialFibrilation',
        'IsDeepVenousThrombosis','IsPeripheralArteryDisease'],
    'comorb_renal': ['IsCrfNoDialysis','IsCrfDialysis'],
    'comorb_hepatico': ['IsCirrhosisChildAB','IsCirrhosisChildC','IsHepaticFailure'],
    'comorb_oncologico': ['IsSolidTumorLocoregional','IsSolidTumorMetastatic',
        'IsHematologicalMalignancy','Is Chemotherapy','Is Radiation Therapy'],
    'comorb_respiratorio_cronico': ['IsSevereCopd','IsAsthma','Is History Of Pneumonia'],
    'comorb_metabolico_endocrino': ['IsDiabetesUncomplicated','IsDiabetesComplicated',
        'IsMorbidObesity','Is Hypothyroidism','Is Hyperthyroidism','Is Dyslipidemias'],
    'comorb_imunossupressao': ['IsImmunossupression','IsSteroidsUse','IsAids'],
    'comorb_transplante': ['Is Solid Organ Transplant','Is Autologous BMT','Is Allogeneic BMT',
        'Is Other Solid Organ Transplant','Is Cardiac Transplant','Is Combined Liverkidney Transplant',
        'Is Combined Pancreaskidney Transplant','Is Liver Transplant','Is Intestinal Transplant',
        'Is Pancreas Transplant','Is Lung Transplant','Is Kidney Transplant'],
    'comorb_neurologico': ['IsStrokeSequelae','IsStrokeNoSequelae','IsDementia'],
    'comorb_psiquiatrico_substancias': ['IsTobaccoConsumption','IsPsychoativeSubstanceDependence',
        'IsAlcoholism','IsPsychiatricDisease'],
    'comorb_outras': ['IsRheumaticDisease','Is Peptic Disease','IsMalnourishment'],
}
TODOS_FLAGS_COMORB = [c for grupo in COMORB_GRUPOS.values() for c in grupo]
faltando = [c for c in TODOS_FLAGS_COMORB if c not in modelo_a.columns]
assert not faltando, f'Comorbidity columns not found: {faltando}'

modelo_a[TODOS_FLAGS_COMORB] = modelo_a[TODOS_FLAGS_COMORB].fillna(0)
modelo_a['n_comorbidades'] = modelo_a[TODOS_FLAGS_COMORB].sum(axis=1)
for grupo, flags in COMORB_GRUPOS.items():
    modelo_a[grupo] = (modelo_a[flags].sum(axis=1) > 0).astype(int)

print('Total comorbidity count — mean:', round(modelo_a['n_comorbidades'].mean(),2),
      '| mediana:', modelo_a['n_comorbidades'].median(), '| max:', modelo_a['n_comorbidades'].max())
print('\nPrevalence by clinical group:')
for grupo in COMORB_GRUPOS:
    print(f'  {grupo:32s} {modelo_a[grupo].mean()*100:5.1f}%')

## 3 · Physiological ratios from the 6-hour extremes

Two ratios, chosen for clinical relevance and for relying on low-missingness components:

- **Shock index** (HR/SBP) — an early marker of hemodynamic compromise, more sensitive than either
  signal alone.
- **Urea/creatinine ratio** — a hint of pre-renal vs. intrinsic renal cause.

The PaO₂/FiO₂ ratio **already comes ready** in Folha6 (`Highest Pa O2Fi O21h`) — no derivation needed.

In [ ]:
modelo_a['shock_index'] = (modelo_a['Highest Heart Rate1h'] /
                            modelo_a['Lowest Systolic Blood Pressure1h'])
modelo_a.loc[modelo_a['Lowest Systolic Blood Pressure1h'] <= 0, 'shock_index'] = np.nan

modelo_a['razao_ureia_creatinina'] = modelo_a['Urea'] / modelo_a['Highest Creatinine1h']
modelo_a.loc[modelo_a['Highest Creatinine1h'] <= 0, 'razao_ureia_creatinina'] = np.nan

for col in ['shock_index','razao_ureia_creatinina']:
    print(f'{col:24s} missingness={modelo_a[col].isna().mean()*100:5.1f}%  '
          f'mediana={modelo_a[col].median():.2f}')

## 4 · Secondary diagnoses (Folha8): from ~130 fields to something tractable

Folha8 carries 12 secondary-diagnosis slots; each has a *"present at ICU admission"* flag
(`IsSecondaryDiagAtICU|AdmissionN`) — exactly what the 24-h temporal contract allows as a predictor.
Only diagnoses flagged as present at admission enter here; anything appearing later during the stay
stays out (leakage).

**The merge here is by position, not by key — and the reason is the same as in Phase 1.** Many cohort
patients (the 551 recovered by positional alignment in Phase 1, plus a few new independent cases visible
only now) have the key (EpimedCode, MedicalRecord) repeated in Folha8 as well. A key-based merge would
mix these patients' information with that of another admission of theirs. We therefore replicate the same
positional-alignment logic here — with the same safety verification: we accept the Folha8 data only when
the exact row matches the expected key; if it does not match, the patient simply receives no secondary
diagnosis (recorded as "none present"), rather than receiving potentially wrong data.

Each slot already comes with a ready-made "major diagnostic area" classification (`Admission Reason Name`),
but with 63 distinct categories — too rich to enter raw. We consolidate into **eleven major system groups**
(cardiovascular, respiratory, infectious, neurological, renal/genitourinary, gastrointestinal/
hepatobiliary, oncological/hematological, metabolic/endocrine, psychiatric, trauma/external causes, and
a residual group for what does not clearly fit a single system).

In [ ]:
N_POSICOES = 12
GRANDE_AREA_MAP = {
    'Cardiovascular':'cardiovascular', 'Cardiorespiratory arrest':'cardiovascular',
    'Choque (exceto sepse)':'cardiovascular', 'Cardiac surgeries (except congenital heart disease)':'cardiovascular',
    'Combined cardiac surgeries':'cardiovascular', 'Cardiac surgeries for congenital heart disease':'cardiovascular',
    'Invasive cardiac and endovascular procedures':'cardiovascular', 'Cirurgias vasculares':'cardiovascular',
    'Procedimentos / cirurgias endovasculares':'cardiovascular', 'Congenital heart disease':'cardiovascular',

    'Respiratory (except sepsis / infection)':'respiratorio', 'Lung, tracheal, and bronchial surgeries':'respiratorio',
    'Other thoracic surgeries':'respiratorio',

    'Infection / sepsis':'infeccioso',

    'Neurological':'neurologico', 'Neurocirurgias':'neurologico', 'Brain death':'neurologico',

    'Renal':'renal_geniturinario', 'Male genital organ diseases':'renal_geniturinario',
    'Female genital organ diseases':'renal_geniturinario', 'Urinary tract surgeries':'renal_geniturinario',
    'Prostate surgeries':'renal_geniturinario', 'Gynecological and breast surgeries':'renal_geniturinario',
    'Cirurgias da mama':'renal_geniturinario', 'Breast diseases':'renal_geniturinario',
    'Obstetrics':'renal_geniturinario', 'Obstetric surgeries and procedures':'renal_geniturinario',

    'Gastrointestinal':'gastro_hepatobiliar', 'Liver, Biliary Tract, Spleen, and Pancreas':'gastro_hepatobiliar',
    'Abdominal / retroperitoneal surgeries':'gastro_hepatobiliar', 'Colon, sigmoid, rectal, and anal canal surgeries':'gastro_hepatobiliar',
    'Abdominal wall hernias and defects':'gastro_hepatobiliar', 'Gastric surgeries':'gastro_hepatobiliar',
    'Cirurgias do intestino delgado':'gastro_hepatobiliar', 'Liver, biliary tract, and pancreas surgeries':'gastro_hepatobiliar',
    'Repair of abdominal wall hernias and defects':'gastro_hepatobiliar', 'Bariatric surgeries':'gastro_hepatobiliar',
    'Esophageal surgeries':'gastro_hepatobiliar',

    'Hematological':'onco_hematologico', 'Tumors and oncological complications':'onco_hematologico',

    'Endocrine / metabolic':'metabolico_endocrino', 'Endocrine gland surgeries':'metabolico_endocrino',

    'Psychiatric diseases':'psiquiatrico',

    'Causas externas':'trauma_externas', 'Non-surgical trauma':'trauma_externas', 'Surgical polytrauma':'trauma_externas',

    'Outros / diversos':'outros', 'Procedimentos invasivos (exceto cardiovascular)':'outros',
    'Cuidados paliativos':'outros', 'Ophthalmological':'outros', 'Outras Cirurgias':'outros',
    'Autoimmune diseases and collagenoses':'outros', 'Multiple organ dysfunction':'outros',
    'Ear, nose, and mouth diseases':'outros', 'Orthopedic surgeries':'outros',
    'Osteoarticular and soft tissue diseases':'outros', 'Cirurgias da pele e tecidos moles':'outros',
    'Head and neck surgeries':'outros', 'Allergic and hypersensitivity reactions':'outros',
    'Skin diseases (except allergic and hypersensitivity reactions)':'outros', 'Monitoring':'outros',
    'Cirurgias de coluna':'outros', 'Otorhinolaryngological surgeries':'outros',
}

reason_cols  = [f'Admission Reason Name{i}' for i in range(1, N_POSICOES+1)]
present_cols = [f'IsSecondaryDiagAtICU|Admission{i}' for i in range(1, N_POSICOES+1)]
cats_reais = set(pd.concat([folha8[c] for c in reason_cols], ignore_index=True).dropna().unique())
nao_mapeadas = cats_reais - set(GRANDE_AREA_MAP.keys())
assert not nao_mapeadas, f'Unmapped categories: {nao_mapeadas}'
print(f'All {len(cats_reais)} "major area" categories covered by the map,',
      f'consolidadas em {len(set(GRANDE_AREA_MAP.values()))} grandes grupos.')

In [ ]:
def is_true(s):
    return s.fillna(0).astype(float).eq(1)

# ---- build records by RAW ROW POSITION (not by key) ----
registros = []
for i in range(1, N_POSICOES+1):
    rcol, pcol = f'Admission Reason Name{i}', f'IsSecondaryDiagAtICU|Admission{i}'
    if rcol not in folha8.columns or pcol not in folha8.columns:
        continue
    presente = is_true(folha8[pcol])
    grupo_i = folha8[rcol].map(GRANDE_AREA_MAP)
    registros.append(pd.DataFrame({'_linha_bruta': folha8['_linha_bruta'], 'presente': presente, 'grupo': grupo_i}))
long_df = pd.concat(registros, ignore_index=True)
long_df = long_df[long_df['presente']]   # only what is present at ICU admission

contagem = long_df.groupby('_linha_bruta').size().rename('n_diag_secund_admissao')
grupos_wide = (long_df.dropna(subset=['grupo']).assign(valor=1)
               .pivot_table(index='_linha_bruta', columns='grupo', values='valor', aggfunc='max', fill_value=0))
grupos_wide.columns = [f'diag2_{g}' for g in grupos_wide.columns]
diag_secund_por_linha = contagem.to_frame().join(grupos_wide, how='outer').reset_index()
diag_cols = [c for c in diag_secund_por_linha.columns if c != '_linha_bruta']

# ---- safety validation: does the Folha8 row really match the expected key? ----
# (same principle as Phase 1 — never blindly trust position without checking)
folha8_chave_por_linha = folha8.set_index('_linha_bruta')[['EpimedCode','MedicalRecord']]
alvo = folha8_chave_por_linha.reindex(modelo_a['_linha_bruta']).reset_index(drop=True)
bate = ((alvo['EpimedCode'].values == modelo_a['EpimedCode'].values) &
        (alvo['MedicalRecord'].values == modelo_a['MedicalRecord'].values))
modelo_a['folha8_integravel'] = bate

n_ok = modelo_a['folha8_integravel'].sum()
n_falha = (~modelo_a['folha8_integravel']).sum()
print(f'Patients with Folha8 correctly aligned by position: {n_ok} of {len(modelo_a)}')
print(f'Patients WITHOUT a safe match in Folha8 (left without secondary diagnoses): {n_falha}')
if n_falha:
    print('\nBreakdown of cases without a safe match:')
    print(modelo_a.loc[~modelo_a['folha8_integravel'], ['EpimedCode','MedicalRecord','_linha_bruta']].to_string(index=False))

# ---- merge by position — only data from rows that passed validation enters ----
modelo_a = modelo_a.merge(diag_secund_por_linha, on='_linha_bruta', how='left', validate='one_to_one')
modelo_a.loc[~modelo_a['folha8_integravel'], diag_cols] = 0     # no safe match -> no diagnosis
modelo_a[diag_cols] = modelo_a[diag_cols].fillna(0)             # no secondary diagnosis present = 0/none

print(f"\nWith >=1 secondary diagnosis present at admission: "
      f"{(modelo_a['n_diag_secund_admissao']>0).sum()} ({(modelo_a['n_diag_secund_admissao']>0).mean()*100:.1f}%)")
print('\nPrevalence by group:')
for c in sorted(diag_cols):
    if c.startswith('diag2_'):
        print(f'  {c:32s} {modelo_a[c].mean()*100:5.1f}%')
# Save misalignment table for audit
if n_falha:
    desalinhados_path = f'{OUTDIR}/fase2_folha8_desalinhamentos.csv'
    modelo_a.loc[~modelo_a['folha8_integravel'],
                 ['EpimedCode','MedicalRecord','_linha_bruta']
    ].to_csv(desalinhados_path, index=False, encoding='utf-8-sig')
    print(f'Tabela de desalinhamentos salva: {desalinhados_path}')


## 5 · Missingness indicators: guided by clinical informativeness

The methodological discussion already made is worth recording here in code: the decision is not about
*how much* is missing, but *why* it is missing. Sisk et al. (2023, *Statistical Methods in Medical
Research*) showed, in simulations geared to clinical prediction models, that missingness indicators
improve performance in many scenarios — but can hurt when missingness depends directly on the outcome,
not just on the patient's overall severity. Groenwold et al. (2012, *CMAJ*) warn against the indicator
method, but in a different context — models estimating the causal effect of an exposure, not predictive
models.

Therefore the indicators enter for **laboratory and physiology** variables (where absence itself is a
clinical signal — the test was not ordered because the patient did not seem sick enough), and **not**
for the boolean comorbidity/complication flags (absence there, at the 0.1% level, is recording noise,
already treated as 0 in earlier sections).

In [ ]:
VARS_INDICADOR = [
    'BMI','Height','Weight',
    'Lowest Systolic Blood Pressure1h','Lowest Diastolic Blood Pressure1h',
    'Lowest Mean Arterial Pressure1h','Highest Heart Rate1h','Highest Respiratory Rate1h',
    'Highest Temperature1h','Lowest Glasgow Coma Scale1h','Highest Leukocyte Count1h',
    'Lowest Platelets Count1h','Highest Creatinine1h','Highest Bilirubin1h',
    'Highest PH1h','Highest Pa O21h','Highest Pa CO21h','Highest Fi O21h',
    'Highest Pa O2Fi O21h','Highest Arterial Lactate1h','Urea','BUN',
    'Charlson Comorbidity Index','Sofa Score',
    'shock_index','razao_ureia_creatinina',
]
VARS_INDICADOR = [c for c in VARS_INDICADOR if c in modelo_a.columns]

indicadores_criados = []
for col in VARS_INDICADOR:
    if modelo_a[col].isna().any():
        novo = f'{col}_ausente'
        modelo_a[novo] = modelo_a[col].isna().astype(int)
        indicadores_criados.append(novo)

print(f'Missingness indicators created: {len(indicadores_criados)}')
for c in indicadores_criados:
    print(f'  {c:44s} {modelo_a[c].mean()*100:5.1f}% ausentes')

# categorical with minimal residual missingness -> explicit label, not a numeric indicator
if 'Chronic Health Status Name' in modelo_a.columns:
    n_falt = modelo_a['Chronic Health Status Name'].isna().sum()
    modelo_a['Chronic Health Status Name'] = modelo_a['Chronic Health Status Name'].astype('string').fillna('Not reported')
    print(f"\n'Chronic Health Status Name': {n_falt} blanks -> labeled 'Not reported'")

### Safety check: is missingness tied to the outcome?

This is exactly the concern Sisk et al. raise. We compare mortality between patients with and without
the datum, for each variable with an indicator.

In [ ]:
checagem = []
for col in VARS_INDICADOR:
    ind = f'{col}_ausente'
    if ind not in modelo_a.columns:
        continue
    tab = pd.crosstab(modelo_a[ind], modelo_a['mortalidade_hospitalar'])
    if tab.shape == (2,2):
        p = stats.chi2_contingency(tab)[1]
        taxa_ausente  = modelo_a.loc[modelo_a[ind]==1, 'mortalidade_hospitalar'].mean()*100
        taxa_presente = modelo_a.loc[modelo_a[ind]==0, 'mortalidade_hospitalar'].mean()*100
        checagem.append((col, taxa_presente, taxa_ausente, p, modelo_a[ind].sum()))
checagem_df = pd.DataFrame(checagem,
    columns=['variable','mort_dado_presente_%','mort_dado_ausente_%','p_valor','n_ausentes']
    ).sort_values('p_valor')
print(checagem_df.round(3).to_string(index=False))

**Reading the two patterns that emerge here** — both expected, for different reasons:

- **Blood gases (PaCO₂, PaO₂, pH, PaO₂/FiO₂):** mortality is *higher* when the test **is present**
  (≈19%) than when absent (≈7–9%). This is the classic informative-missingness pattern "on the right
  side": arterial blood gas is ordered for more unstable patients — its absence reflects lower severity,
  not a problem.
- **Blood pressure (SBP/DBP/MAP) and shock index:** the pattern reverses, and far more strongly —
  mortality of ≈60% among the few patients (≈0.5%) without these vital signs recorded, versus ≈14%
  among the rest. The most likely reading is not a post-outcome recording artifact (the kind of
  missingness Sisk et al. consider risky): it is that, in a patient in frank hemodynamic collapse or
  arrest, blood pressure simply is not measurable — information clinically available *at the moment*,
  not discovered afterwards. Even so, with such a small sample (≈25 patients), this indicator deserves
  caution in the modeling stage — watching whether SHAP or the coefficients treat it as a
  disproportionate shortcut, and considering stronger regularization there if needed. There is no reason,
  however, to drop it now: it does not fit the pattern the literature flags as problematic.

## 6 · Table 1 — descriptive by outcome (hospital discharge vs. death)

A representative sample of the full Table 1 — age, severity scores, the new comorbidity count, secondary
diagnoses present at admission, and some physiological markers. The manuscript's final table should
expand this list to cover all variables of interest, but the construction logic (median [IQR] +
Mann-Whitney for continuous, n (%) + chi-square for binary) is already ready to be reapplied to any
variable in the base.

In [ ]:
def linha_continua(df, col, grupo, nome=None):
    a = df.loc[df[grupo]==0, col].dropna(); b = df.loc[df[grupo]==1, col].dropna()
    p = stats.mannwhitneyu(a, b, alternative='two-sided').pvalue if len(a) and len(b) else np.nan
    return {'Variable': nome or col,
            'Discharge (median [IQR])': f'{a.median():.1f} [{a.quantile(.25):.1f}\u2013{a.quantile(.75):.1f}]',
            'Death (median [IQR])': f'{b.median():.1f} [{b.quantile(.25):.1f}\u2013{b.quantile(.75):.1f}]',
            'p-valor': p}

def linha_binaria(df, col, grupo, nome=None):
    a = df.loc[df[grupo]==0, col].dropna(); b = df.loc[df[grupo]==1, col].dropna()
    tab = [[a.sum(), len(a)-a.sum()], [b.sum(), len(b)-b.sum()]]
    p = stats.chi2_contingency(tab)[1]
    return {'Variable': nome or col,
            'Discharge (median [IQR])': f'{a.sum():.0f} ({a.mean()*100:.1f}%)',
            'Death (median [IQR])': f'{b.sum():.0f} ({b.mean()*100:.1f}%)',
            'p-valor': p}

grupo = 'mortalidade_hospitalar'
linhas = [
    linha_continua(modelo_a, 'Age', grupo, 'Idade, anos'),
    linha_continua(modelo_a, 'BMI', grupo, 'IMC'),
    linha_continua(modelo_a, 'Saps3Points', grupo, 'SAPS3, pontos'),
    linha_continua(modelo_a, 'Sofa Score', grupo, 'SOFA, pontos'),
    linha_continua(modelo_a, 'Charlson Comorbidity Index', grupo, 'Charlson, pontos'),
    linha_continua(modelo_a, 'n_comorbidades', grupo, 'Nº de comorbidades (contagem)'),
    linha_continua(modelo_a, 'n_diag_secund_admissao', grupo, 'No. of secondary diagnoses at admission'),
    linha_continua(modelo_a, 'shock_index', grupo, 'Shock index (FC/PAS)'),
    linha_continua(modelo_a, 'Highest Arterial Lactate1h', grupo, 'Lactato, mmol/L'),
    linha_continua(modelo_a, 'Lowest Glasgow Coma Scale1h', grupo, 'Glasgow (menor, 6h)'),
]
for c, nome in [('comorb_cardiovascular','Comorbidade cardiovascular'),
                ('comorb_renal','Comorbidade renal'),
                ('comorb_oncologico','Oncological comorbidity'),
                ('diag2_infeccioso','Diag. secund. \u2014 infeccioso'),
                ('diag2_cardiovascular','Diag. secund. \u2014 cardiovascular')]:
    linhas.append(linha_binaria(modelo_a, c, grupo, nome))

tabela1 = pd.DataFrame(linhas)
tabela1['p-valor'] = tabela1['p-valor'].apply(lambda p: '<0.001' if p < 0.001 else f'{p:.3f}')
print(tabela1.to_string(index=False))

tabela1_xlsx_path = f'{OUTDIR}/modelo_A_tabela1.xlsx'
tabela1.to_excel(tabela1_xlsx_path, index=False)
print(f'Tabela 1 salva: {tabela1_xlsx_path}')

## 7 · The two variable sets of Model A

The count variables and
comorbidity groups, secondary-diagnosis groups, physiological ratios, and missingness indicators are
**raw components**, not ready-made scores — and therefore belong to **Set 2**.
The architecture document is specific: Set 1 exists to isolate *only* what the ready-made scores
(SAPS3, SOFA, Charlson, MFI) already summarize; Set 2 exists to isolate *only* the granular components —
exactly the contrast that answers the study question. Mixing the two in Set 1 would erase that dividing
line.

- **Set 1 (ready-made scores):** demographics and admission context + Charlson, MFI (points/Score/Frail),
  SAPS3 points, SOFA.
- **Set 2 (raw components):** the same demographics and admission context + comorbidities (count and
  groups) + acute admission reasons (Folha4) + first-hour and first-day complications (Folha5) + 6-hour
  physiology and laboratory + the two derived ratios + secondary diagnoses (count and groups) +
  missingness indicators.

In [ ]:
DEMOGRAFIA_CONTEXTO = ['Age','Gender','Weight','Height','BMI','AdmissionSourceName',
    'AdmissionTypeName','AdmissionReasonName','AdmissionMainDiagnosisName','Month Year',
    'Chronic Health Status Name','Status COVID 19','OrigemPagamento','BMI_ausente','Height_ausente','Weight_ausente']

ESCORES_PRONTOS = ['Charlson Comorbidity Index','MFI points','MFI Score','Frail Patient MFI',
    'Saps3Points','Sofa Score','Charlson Comorbidity Index_ausente','Sofa Score_ausente']

COMORB_GRUPOS_NOMES = list(COMORB_GRUPOS.keys())
DIAG2_GRUPOS_NOMES  = [c for c in modelo_a.columns if c.startswith('diag2_')]

FOLHA4_RAZOES = ['IsNeurologicalComaStuporObtundedDelirium','IsNeurologicalSeizures',
    'IsNeurologicalFocalNeurologicDeficit','IsNeurologicalIntracranialMassEffect',
    'IsCardiovascularHypovolemicHemorrhagicShock','IsCardiovascularSepticShock',
    'IsCardiovascularRhythmDisturbances','IsCardiovascularAnaphylacticMixedUndefinedShock',
    'IsDigestiveAcuteAbdomen','IsDigestiveSeverePancreatitis','IsLiverFailure',
    'IsTransplantSolidOrgan','IsTraumaMultipleTrauma','IsCardiacSurgery','IsNeurosurgery',
    'Is Pneumonia','Is Infection Nosocomial']

FOLHA5_COMPLICACOES = ['IsRespiratoryFailure1h','IsMechanicalVentilation1h','IsNonInvasiveVentilation1h',
    'IsVasopressors1h','IsCardiacArrhythmias1h','IsCardiopulmonaryArrest1h','IsAcuteKidneyInjury1h',
    'IsRenalReplacementTherapy1h','IsGastrointestinalBleeding1h','IsIntracranialMassEffect1h',
    'IsNeutropenia1h','IsAsystole1h','IsPulselessElectricalActivity1h',
    'IsVentricularSustainedCardiopulmonary1h','IsAcuteAtrialFibrilation1h','IsAtrialFlutter1h',
    'IsVentricularSustainedArrhythmia1h',
    'IsRespiratoryFailure','IsMechanicalVentilation','IsNonInvasiveVentilation','IsVasopressors',
    'IsCardiacArrhythmias','IsCardiopulmonaryArrest','IsAcuteKidneyInjury','IsRenalReplacementTherapy',
    'IsGastrointestinalBleeding','IsIntracranialMassEffect','IsNeutropenia','IsAsystole',
    'IsPulselessElectricalActivity','IsVentricularSustainedCardiopulmonary','IsAcuteAtrialFibrilation',
    'IsAtrialFlutter','IsVentricularSustainedArrhythmia']

FISIO6H_RAW = ['Lowest Systolic Blood Pressure1h','Lowest Diastolic Blood Pressure1h',
    'Lowest Mean Arterial Pressure1h','Highest Heart Rate1h','Highest Respiratory Rate1h',
    'Highest Temperature1h','Lowest Glasgow Coma Scale1h','Highest Leukocyte Count1h',
    'Lowest Platelets Count1h','Highest Creatinine1h','Highest Bilirubin1h','Highest PH1h',
    'Highest Pa O21h','Highest Pa CO21h','Highest Fi O21h','Highest Pa O2Fi O21h',
    'Highest Arterial Lactate1h','Urea','BUN']

IND_LAB_FISIO = [f'{c}_ausente' for c in FISIO6H_RAW] + ['shock_index_ausente','razao_ureia_creatinina_ausente']

CONJUNTO_1_ESCORES = [c for c in DEMOGRAFIA_CONTEXTO + ESCORES_PRONTOS if c in modelo_a.columns]
CONJUNTO_2_BRUTO = [c for c in (DEMOGRAFIA_CONTEXTO + ['n_comorbidades'] + COMORB_GRUPOS_NOMES +
    FOLHA4_RAZOES + FOLHA5_COMPLICACOES + FISIO6H_RAW +
    ['shock_index','razao_ureia_creatinina'] +
    ['n_diag_secund_admissao'] + DIAG2_GRUPOS_NOMES + IND_LAB_FISIO) if c in modelo_a.columns]

print(f'Set 1 (ready-made scores + demographics/context):    {len(CONJUNTO_1_ESCORES)} variables')
print(f'Set 2 (raw components + demographics/context):       {len(CONJUNTO_2_BRUTO)} variables')
print(f'Overlap (demographics/context only, expected):       {len(set(CONJUNTO_1_ESCORES) & set(CONJUNTO_2_BRUTO))} columns')

## 8 · Save the Phase 2 base

The complete base, with all engineered variables, plus the two column lists (Set 1 and Set 2) saved
separately — so the modeling stage can read them directly, without recomputing anything.

In [ ]:
# OUTDIR defined in the configuration cell (§1) — we only make sure it exists.
os.makedirs(OUTDIR, exist_ok=True)

fase2_path = f'{OUTDIR}/modelo_A_fase2.csv'
modelo_a.to_csv(fase2_path, index=False, encoding='utf-8-sig')

import json
sets_path = f'{OUTDIR}/modelo_A_sets_variaveis.json'
with open(sets_path, 'w', encoding='utf-8') as f:
    json.dump({'set_1_escores': CONJUNTO_1_ESCORES,
               'set_2_bruto': CONJUNTO_2_BRUTO,
               'benchmark': 'Saps3Death Probability Standard Equation',
               'desfechos': ['mortalidade_hospitalar','mortalidade_uti']}, f, ensure_ascii=False, indent=2)

print(f'Base da Fase 2 salva: {fase2_path}  ({modelo_a.shape[0]} linhas x {modelo_a.shape[1]} colunas)')
print(f'Variable sets saved: {sets_path}')

---
### What this phase delivered
- Comorbidities consolidated: total count + eleven clinical groups, from the ~53 Folha3 flags.
- Two physiological ratios derived (shock index, urea/creatinine); PaO₂/FiO₂ kept as it already came.
- **Folha8 integrated for the first time, by positional alignment (not by key)** — replicating the same
  Phase 1 logic, with the same safety verification. This revealed a finding specific to this sheet: 5
  patients who were unique pairs and had never shown any problem (in Folha3–6) showed misalignment only
  in Folha8 — the same "neighbor swap" signature documented for the original 555, but in a different
  sheet, affecting different patients. These 5 are left without secondary diagnoses (treated as "none
  present"), preserving the rest of their data intact.
- Missingness indicators created for laboratory and physiology, guided by clinical informativeness —
  with a check that confirmed the expected pattern in blood gases, and flagged one finding (absent blood
  pressure) for careful observation in the modeling stage, without justifying exclusion now.
- Table 1 (sample) built, with the logic ready to expand to all variables of interest.
- The two variable sets defined and saved — Set 1 faithful to its "scores only" role, Set 2 faithful to
  its "raw components only" role — now including `OrigemPagamento` (Public/SUS, Self-pay, Private
  insurance, Not reported), the new variable derived in Phase 1.

### Next steps (Model A modeling)
- One-hot encode categoricals in each set — `OrigemPagamento` joins the categorical list alongside
  `Gender`, `AdmissionReasonName`, etc.
- Impute missing values (median or similar) in continuous variables — the indicators already created
  accompany the imputation.
- Patient-level train/test split and temporal validation (`Month Year` / `adm_datetime`).
- Model from simple to powerful in each set: regularized logistic regression → gradient boosting → SHAP,
  always benchmarked against SAPS3 via the DeLong test.